# 01 — Exploration de PaySim

**Objectif :** comprendre les données avant de construire quoi que ce soit (issue #6).

- **Source :** [PaySim](https://www.kaggle.com/datasets/ealaxi/paysim1), transactions mobile money
  **simulées** (aucune donnée réelle). 6 362 620 transactions sur 743 heures (`step`).
- **Méthode :** l'exploration se fait sur un échantillon de 10 % (`random_state=42`) pour aller vite ;
  le nettoyage sera appliqué au fichier complet (section 7).
- Les sorties de ce notebook sont retirées au commit par nbstripout : il faut l'exécuter pour voir
  les résultats.

## 1. Chargement

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Le notebook est dans ml-service/notebooks/ : on rend importable le package app/.
RACINE = Path.cwd().resolve()
if RACINE.name == "notebooks":
    RACINE = RACINE.parent
sys.path.insert(0, str(RACINE))

from app.donnees import charger_paysim  # noqa: E402

# Couleurs validées (contraste et daltonisme) : bleu = transactions normales, orange = fraudes.
BLEU, ORANGE = "#2a78d6", "#eb6834"
plt.rcParams.update({
    "figure.figsize": (9, 4),
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.color": "#e5e4e0",
    "axes.axisbelow": True,
    "axes.titleweight": "bold",
})
pd.set_option("display.float_format", "{:,.2f}".format)

In [ ]:
df = charger_paysim(frac=0.1, random_state=42)

print(f"{len(df):,} transactions, {int(df['isFraud'].sum()):,} fraudes "
      f"({df['isFraud'].mean():.3%})")
print(f"Mémoire : {df.memory_usage(deep=True).sum() / 1e6:.0f} Mo")
df.head()

In [ ]:
df.dtypes

**Constat.** 636 262 transactions, dont 817 fraudes (0,13 %, environ 1 sur 780) : la classe fraude est très minoritaire. C'est pour cela que l'accuracy est inutile ici : un modèle qui répond toujours « normale » obtiendrait 99,87 % d'accuracy sans détecter une seule fraude. On suivra la PR-AUC, le rappel et la précision sur la classe fraude. Les types de colonnes sont fixés par `charger_paysim()` : l'échantillon tient en ~60 Mo.

## 2. Types de transaction

In [ ]:
repartition = (
    df["type"].value_counts()
    .to_frame("transactions")
    .assign(part=lambda t: t["transactions"] / t["transactions"].sum())
)
repartition.style.format({"transactions": "{:,}", "part": "{:.1%}"})

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3))
donnees = repartition.sort_values("transactions")
barres = ax.barh(donnees.index.astype(str), donnees["transactions"], color=BLEU, height=0.6)
ax.bar_label(barres, labels=[f"{p:.1%}" for p in donnees["part"]], padding=4)
ax.set_title("Nombre de transactions par type (échantillon 10 %)")
ax.set_xlabel("Transactions")
ax.xaxis.set_major_formatter(lambda x, _: f"{x:,.0f}".replace(",", " "))
ax.grid(axis="y", visible=False)
plt.show()

**Constat.** CASH_OUT (35 %) et PAYMENT (34 %) dominent, suivis de CASH_IN (22 %). TRANSFER ne représente que 8 % des transactions et DEBIT moins de 1 %.

## 3. Montants

Les montants vont de quelques unités à plusieurs millions : en échelle normale, presque toutes
les transactions seraient tassées dans la première barre. On utilise donc une **échelle
logarithmique** (chaque graduation = ×10).

In [ ]:
print(f"Montants nuls : {(df['amount'] == 0).sum()}")
df.groupby("type", observed=True)["amount"].describe()

In [ ]:
positifs = df[df["amount"] > 0]  # log(0) n'existe pas
classes = np.logspace(np.log10(positifs["amount"].min()), np.log10(positifs["amount"].max()), 60)

fig, ax = plt.subplots()
ax.hist(positifs["amount"], bins=classes, color=BLEU, edgecolor="white", linewidth=0.5)
ax.set_xscale("log")
ax.set_title("Distribution des montants (échelle logarithmique)")
ax.set_xlabel("Montant")
ax.set_ylabel("Transactions")
plt.show()

In [ ]:
types = repartition.index.astype(str).tolist()
fig, ax = plt.subplots()
ax.boxplot(
    [positifs.loc[positifs["type"] == t, "amount"] for t in types],
    tick_labels=types, whis=(5, 95), showfliers=False,
    medianprops={"color": ORANGE, "linewidth": 2},
)
ax.set_yscale("log")
ax.set_title("Montants par type (boîte : 25e–75e percentile, moustaches : 5e–95e)")
ax.set_ylabel("Montant (échelle log)")
plt.show()

In [ ]:
# Comparaison fraude / normale sur les seuls types où la fraude existe (voir section 4),
# en part de chaque groupe (chaque courbe somme à 100 %) : les deux groupes n'ont pas du tout
# la même taille. density=True serait trompeur ici : avec des classes logarithmiques, il divise
# par la largeur de classe et gonfle artificiellement les petits montants.
cibles = positifs[positifs["type"].isin(["TRANSFER", "CASH_OUT"])]

fig, ax = plt.subplots()
for valeur, nom, couleur in [(0, "Normale", BLEU), (1, "Fraude", ORANGE)]:
    montants = cibles.loc[cibles["isFraud"] == valeur, "amount"]
    ax.hist(montants, bins=classes, weights=np.full(len(montants), 1 / len(montants)),
            histtype="step", linewidth=2, color=couleur, label=nom)
ax.set_xscale("log")
ax.set_title("Montants des fraudes et des transactions normales (TRANSFER et CASH_OUT)")
ax.set_xlabel("Montant (échelle log)")
ax.set_ylabel("Part du groupe")
ax.yaxis.set_major_formatter(lambda y, _: f"{y:.0%}")
ax.legend(frameon=False)
plt.show()

cibles.groupby("isFraud")["amount"].describe()

In [ ]:
# PaySim plafonne les montants à 10 000 000 : est-ce propre aux fraudes ?
plafond = cibles["amount"] == 10_000_000
cibles[plafond].groupby("isFraud").size().rename("transactions à 10 000 000 exactement")

**Constat.** Les montants s'étalent sur plusieurs ordres de grandeur (de quelques centimes à 69 millions), d'où l'échelle logarithmique. Ils dépendent fortement du type : médiane ≈ 3 000 pour DEBIT, ≈ 9 500 pour PAYMENT, ≈ 145 000 pour CASH_IN et CASH_OUT, ≈ 490 000 pour TRANSFER.

Sur TRANSFER et CASH_OUT, les fraudes portent sur des montants plus élevés (médiane ≈ 450 000 contre ≈ 172 000), mais les deux distributions se recouvrent largement : **le montant seul ne suffira pas** à détecter la fraude.

Le simulateur plafonne les montants à 10 000 000, pour les fraudes (20) comme pour les transactions normales (300) : ce plafond n'est pas un signal de fraude. Une seule transaction a un montant nul (une fraude CASH_OUT), à examiner avec les soldes.

## 4. Taux de fraude par type

In [ ]:
par_type = (
    df.groupby("type", observed=True)["isFraud"]
    .agg(transactions="size", fraudes="sum", taux="mean")
    .sort_values("fraudes", ascending=False)
)
par_type.style.format({"transactions": "{:,}", "fraudes": "{:,}", "taux": "{:.3%}"})

In [ ]:
types_fraude = set(par_type.index[par_type["fraudes"] > 0].astype(str))
assert types_fraude == {"TRANSFER", "CASH_OUT"}, f"Types avec fraude inattendus : {types_fraude}"
print("Vérifié : la fraude ne concerne que TRANSFER et CASH_OUT.")

**Constat.** La fraude ne concerne que **TRANSFER** (415 fraudes, 0,78 % des virements) et **CASH_OUT** (402 fraudes, 0,18 % des retraits) ; les trois autres types n'en ont aucune (vérifié par `assert`). C'est le scénario simulé par PaySim : le fraudeur prend le contrôle d'un compte, vire l'argent vers un compte qu'il contrôle (TRANSFER), puis le retire (CASH_OUT).

Piste pour le notebook 02 : n'entraîner le modèle que sur ces deux types. Décision à prendre en 02, pas ici.